# Tentativo 19 · 2. Lo split

**Uno split, generato una volta sola qui, per i due compiti.** Le testate si ordinano per **voto esatto** (a parità di
voto, a caso con il seme `SEME_SPLIT` = 1) e, a blocchi di 5 consecutive, ognuna va in un fold diverso
(`modelli.crea_split`). Il risultato si salva in `split.csv`, e tutti i notebook successivi leggono solo quel file.
Con lo stesso seme è identico al primo split del tentativo 18, così i risultati restano confrontabili.

**Perché uno solo, per voto.** I livelli sono fasce di voto consecutive, quindi lo split è bilanciato anche per livello
(al più una testata di differenza tra i fold), e in più è bilanciato **dentro** i livelli, cosa che lo split per livello
del tentativo 16 non fa. Con le stesse testate negli stessi fold, classificatore e regressore si confrontano a coppie.
Una testata non è mai divisa tra fold; i gruppi di proprietà sono già risolti nel tentativo 16.

In [1]:
import pandas as pd

import anonimizza as A
import modelli as Mo

t = pd.read_csv(A.DATASET / "testate.csv", index_col="domain")
fold = Mo.crea_split(t.score, Mo.SEME_SPLIT)
fold.rename("fold").rename_axis("domain").to_csv(A.DATASET / "split.csv")
print(f"scritto split.csv: {len(t)} testate, 5 fold, seme {Mo.SEME_SPLIT}")
diciotto = pd.read_csv(A.DATI / "tentativi" / "18_PunteggioEsatto" / "split.csv", index_col=0).split1.reindex(t.index)
print("identico al primo split del tentativo 18:", bool((fold == diciotto).all()))

scritto split.csv: 226 testate, 5 fold, seme 1
identico al primo split del tentativo 18: True


**Testate per livello in ogni fold**: al più una di differenza.

In [2]:
pd.crosstab(fold.rename("fold"), t.livello)

livello,l1,l2,l3,l4,l5
fold,,,,,
1,7,9,5,16,8
2,7,9,6,16,7
3,7,8,6,16,8
4,7,9,6,15,9
5,7,8,7,16,7


**Il voto in ogni fold**: stessa media e dispersione, e lo stesso voto medio dentro i livelli. Per confronto lo split
per livello del tentativo 16, dove il voto medio delle l1 andava da 12 a 31 a seconda del fold.

In [3]:
livello16 = pd.read_csv(A.ORIGINE / "testate.csv", index_col="domain").fold.reindex(t.index)
righe = []
for nome, f in [("per voto (questo)", fold), ("per livello (tentativo 16)", livello16)]:
    for k in range(1, 6):
        g = t[f == k]
        righe.append({"split": nome, "fold": k, "testate": len(g), "voto medio": g.score.mean(), "dev. std": g.score.std(),
                      "voto medio l1": g.score[g.livello == "l1"].mean(), "voto medio l4": g.score[g.livello == "l4"].mean()})
pd.DataFrame(righe).set_index(["split", "fold"]).round(1)

testate  voto medio  dev. std  voto medio l1  \
split                      fold                                                 
per voto (questo)          1          45        68.4      27.2           21.6   
                           2          45        68.0      27.5           19.9   
                           3          45        68.6      27.1           21.4   
                           4          46        69.2      27.5           20.6   
                           5          45        68.3      27.5           19.9   
per livello (tentativo 16) 1          46        67.6      28.1           18.5   
                           2          45        69.8      24.5           31.4   
                           3          45        68.6      27.5           19.6   
                           4          45        69.8      27.4           21.8   
                           5          45        66.7      29.0           12.1   

                                 voto medio l4  
split                      fold                 
per voto (questo)          1              84.3  
                           2              85.5  
                           3              84.3  
                           4              85.3  
                           5              85.6  
per livello (tentativo 16) 1              84.3  
                           2              86.8  
                           3              84.1  
                           4              86.8  
                           5              82.8